# Silver to Gold transformation — final

This is a targeted final version of the Phase 5 notebook. It keeps the same five Gold datasets, corrects metric grain issues, validates each output, and publishes them to MinIO.

In [ ]:
import os

from dotenv import load_dotenv
from pyspark.sql import functions as F

from src.spark_minio import get_spark_session

load_dotenv()
bucket_name = os.environ["MINIO_BUCKET"]
spark = get_spark_session("sanlam-silver-to-gold-final")
spark.sparkContext.setLogLevel("WARN")

SILVER_ROOT = f"s3a://{bucket_name}/silver"
SILVER_TABLES = ["customers", "agents", "insurance_products", "policies", "policy_coverages", "policy_status_history", "premium_transactions", "claims", "claim_payments"]
silver = {name: spark.read.parquet(f"{SILVER_ROOT}/{name}/") for name in SILVER_TABLES}

customers = silver["customers"]
agents = silver["agents"]
insurance_products = silver["insurance_products"]
policies = silver["policies"]
policy_coverages = silver["policy_coverages"]
policy_status_history = silver["policy_status_history"]
premium_transactions = silver["premium_transactions"]
claims = silver["claims"]
claim_payments = silver["claim_payments"]

## Shared aggregates

Aggregate one-to-many payment and coverage data before joining it to claims or policies. This preserves the intended grain and prevents sums being multiplied.

In [ ]:
paid_payments_by_claim = (
    claim_payments.filter(F.col("payment_status") == "PAID")
    .groupBy("claim_id")
    .agg(
        F.count("*").alias("payment_count"),
        F.sum("payment_amount").alias("total_paid_amount"),
    )
)

claims_by_policy = (
    claims.groupBy("policy_id")
    .agg(
        F.count("*").alias("claim_count"),
        F.sum("claimed_amount").alias("total_claimed_amount"),
        F.sum(F.coalesce(F.col("approved_amount"), F.lit(0))).alias("total_approved_amount"),
    )
)

payments_by_policy = (
    claims.select("claim_id", "policy_id")
    .join(paid_payments_by_claim, "claim_id", "left")
    .groupBy("policy_id")
    .agg(
        F.sum(F.coalesce(F.col("payment_count"), F.lit(0))).alias("number_of_payments"),
        F.sum(F.coalesce(F.col("total_paid_amount"), F.lit(0))).alias("total_paid_amount"),
    )
)

coverage_by_policy = policy_coverages.groupBy("policy_id").agg(F.sum("coverage_limit").alias("total_coverage_limit"))

premium_by_policy = (
    premium_transactions.groupBy("policy_id")
    .pivot("transaction_status", ["PAID", "FAILED", "PENDING", "REVERSED"])
    .agg(F.count("*").alias("count"), F.sum("amount").alias("total_amount"))
    .fillna(0)
)

In [ ]:
# 1. One row per claim
claims_performance = (
    claims.join(policies.select("policy_id", "customer_id", "product_id", "policy_status"), "policy_id", "left")
    .join(customers.select("customer_id", "customer_segment", "province"), "customer_id", "left")
    .join(insurance_products.select("product_id", "product_name", "product_line"), "product_id", "left")
    .join(paid_payments_by_claim, "claim_id", "left")
    .fillna({"payment_count": 0, "total_paid_amount": 0})
)

# 2. One row per due month. Keep the month as a date so ordering stays chronological.
premium_collection = (
    premium_transactions.withColumn("due_month", F.trunc("due_date", "month"))
    .groupBy("due_month")
    .pivot("transaction_status", ["PAID", "FAILED", "PENDING", "REVERSED"])
    .agg(F.count("*").alias("count"), F.sum("amount").alias("total_amount"))
    .fillna(0)
    .withColumn("collection_rate_via_count", F.when((F.col("PAID_count") + F.col("FAILED_count") + F.col("PENDING_count") + F.col("REVERSED_count")) > 0, F.col("PAID_count") / (F.col("PAID_count") + F.col("FAILED_count") + F.col("PENDING_count") + F.col("REVERSED_count"))))
    .withColumn("collection_rate_via_amount", F.when((F.col("PAID_total_amount") + F.col("FAILED_total_amount") + F.col("PENDING_total_amount") + F.col("REVERSED_total_amount")) > 0, F.col("PAID_total_amount") / (F.col("PAID_total_amount") + F.col("FAILED_total_amount") + F.col("PENDING_total_amount") + F.col("REVERSED_total_amount"))))
    .orderBy("due_month")
)

In [ ]:
premium_collection.limit(5).toPandas()

In [ ]:
# 3. One row per policy
policy_performance = (
    policies.select("policy_id", "policy_number", "customer_id", "product_id", "agent_id", "policy_status", "payment_frequency", "premium_amount", "sum_insured", "underwriting_risk_band")
    .join(customers.select("customer_id", "customer_segment", "province"), "customer_id", "left")
    .join(agents.select("agent_id", "agent_code", "agent_name", "region", "channel"), "agent_id", "left")
    .join(insurance_products.select("product_id", "product_name", "product_line"), "product_id", "left")
    .join(coverage_by_policy, "policy_id", "left")
    .join(claims_by_policy, "policy_id", "left")
    .join(payments_by_policy, "policy_id", "left")
    .join(premium_by_policy, "policy_id", "left")
    .fillna(0)
)

# 4. One row per customer
customer_base = (
    customers.select("customer_id", "date_of_birth", "gender", "province", "customer_segment", "risk_score", "has_email")
    .withColumn("age_years", (F.datediff(F.current_date(), "date_of_birth") / F.lit(365.25)).cast("decimal(4,1)"))
    .withColumn("age_bin", F.when(F.col("age_years") < 25, "18-24").when(F.col("age_years") < 35, "25-34").when(F.col("age_years") < 45, "35-44").when(F.col("age_years") < 55, "45-54").when(F.col("age_years") < 65, "55-64").otherwise("65+"))
    .drop("date_of_birth")
)
customer_metrics = (
    policies.select("policy_id", "customer_id", "policy_status", "premium_amount")
    .join(claims_by_policy, "policy_id", "left")
    .join(payments_by_policy, "policy_id", "left")
    .groupBy("customer_id")
    .agg(
        F.count("policy_id").alias("number_of_policies"),
        F.sum(F.when(F.col("policy_status") == "ACTIVE", 1).otherwise(0)).alias("number_of_active_policies"),
        F.sum("premium_amount").alias("total_policy_premium_amount"),
        F.sum(F.coalesce(F.col("claim_count"), F.lit(0))).alias("number_of_claims"),
        F.sum(F.coalesce(F.col("total_claimed_amount"), F.lit(0))).alias("total_claimed_amount"),
        F.sum(F.coalesce(F.col("total_approved_amount"), F.lit(0))).alias("total_approved_amount"),
        F.sum(F.coalesce(F.col("number_of_payments"), F.lit(0))).alias("number_of_payments"),
        F.sum(F.coalesce(F.col("total_paid_amount"), F.lit(0))).alias("total_paid_amount"),
    )
)
customer_360 = customer_base.join(customer_metrics, "customer_id", "left").fillna(0)

In [ ]:
# 5. One row per policy-status event
policy_status_history_gold = (
    policy_status_history.join(policies.select("policy_id", "inception_date", "product_id", "customer_id"), "policy_id", "left")
    .join(insurance_products.select("product_id", "product_line"), "product_id", "left")
    .withColumn("change_month", F.trunc("changed_at", "month"))
    .withColumn("days_since_inception", F.datediff("changed_at", "inception_date"))
    .withColumn("is_lapse_or_cancel", F.col("new_status").isin("LAPSED", "CANCELLED"))
)

gold = {
    "claims_performance": (claims_performance, "claim_id"),
    "premium_collection": (premium_collection, "due_month"),
    "policy_performance": (policy_performance, "policy_id"),
    "customer_360": (customer_360, "customer_id"),
    "policy_status_history": (policy_status_history_gold, "policy_status_history_id"),
}

for table_name, (dataframe, grain_key) in gold.items():
    duplicates = dataframe.groupBy(grain_key).count().filter(F.col("count") > 1).count()
    if duplicates:
        raise ValueError(f"{table_name} violates its grain ({grain_key}): {duplicates} duplicate key(s)")
    print(f"Validated {table_name}: {dataframe.count()} row(s), one row per {grain_key}")

In [ ]:
# Publish only after the validation cell above has completed successfully.
GOLD_ROOT = f"s3a://{bucket_name}/gold"

for table_name, (dataframe, _) in gold.items():
    output_path = f"{GOLD_ROOT}/{table_name}/"
    dataframe.write.mode("overwrite").option("compression", "snappy").parquet(output_path)
    print(f"Published {table_name}: {dataframe.count()} rows -> {output_path}")

print("Gold publishing completed successfully.")

In [ ]:
import pandas as pd 
pd.set_option('display.max_columns', None)

In [ ]:
premium_collection.limit(5).toPandas()